# Controlled Agentic RAG for Network Operations
Upgrade of the plain RAG prototype. Same corpus, same Qdrant, same local Ollama model; only the **control flow** changes.

```
question -> [guard] -> [plan / route] -> [retrieve per source] -> [grade evidence]
                                                 ^                    |  enough -> [answer] -> response + sources
                                                 |  not enough        v
                                          [rewrite query]  (max 1)    else -> "I don't know"
```

| Step | Type | Purpose |
|---|---|---|
| guard | code | refuse requests for our passwords, keys, IP plans, tickets, prompt injection |
| plan | LLM | split cross-source questions; choose the source for each part |
| retrieve | code (tool) | read-only Qdrant search, filtered by classification, status **and role** |
| grade | LLM | is the evidence enough to answer? |
| rewrite | LLM | one better query, then stop (loop cap) |
| answer | LLM | grounded answer; sources are attached by code |

**Needs:** Docker/Qdrant running with your collection loaded, Ollama running, and `pip install langgraph`.

In [ ]:
%pip install -q langgraph

## 1. Check the system

In [ ]:
import rag_demo, agentic_rag as ag

store = rag_demo.get_vector_store()
n = store.client.count(rag_demo.COLLECTION).count
print(f"Collection '{rag_demo.COLLECTION}': {n} chunks | model: {ag.CHAT_MODEL} | context window: {ag.NUM_CTX}")
assert n > 0, "Collection is empty: run the ingestion section of Governed_RAG_LLM_Prototype.ipynb first."

## 2. The workflow as a graph
Paste the text below into https://mermaid.live to get a figure for the thesis.

In [ ]:
print(ag.graph_mermaid())

## 3. Single-source question
Watch the **PATH** line: each step is also a separate span in LangSmith when tracing is on.

In [ ]:
ag.show(ag.run_agent("How can I explicitly configure the BGP router ID in SONiC?"))

## 4. Cross-source question (the agent splits it)
Expect two sub-queries: one for OpenConfig, one for SONiC, and sources from both products.

In [ ]:
ag.show(ag.run_agent("How is BGP modelled in OpenConfig, and how does SONiC choose the BGP router ID?"))

## 5. Controls: the agent should refuse
**a)** sensitive request (stopped by code, zero LLM calls) **b)** a question the corpus cannot answer **c)** a role with no access.

In [ ]:
ag.show(ag.run_agent("What is the admin password of our production core router?"))

In [ ]:
ag.show(ag.run_agent("How do I configure OSPF areas on Juniper Junos?"))

In [ ]:
ag.show(ag.run_agent("How can I explicitly configure the BGP router ID in SONiC?", role="guest"))

## 6. Optional: LangSmith tracing
Turn it on, then re-run any question above. Every graph step appears as its own span, tagged with the role and graph version.
Public corpus only. The key is typed at a hidden prompt.

In [ ]:
if ag.enable_tracing():
    ag.show(ag.run_agent("How is BGP modelled in OpenConfig, and how does SONiC choose the BGP router ID?", tags=["demo"]))

## 7. Plain RAG vs agentic RAG
Same questions, same model, same retrieval tool. Start with the **quick subset** (6 questions), then run the full set.
A local 3B model on CPU is slow: the agent makes about 3 LLM calls per question, plain RAG makes 1.

In [ ]:
quick = [q for q in ag.EVAL_SET if q["id"] in (3, 5, 7, 9, 13, 16)]
df, summary = ag.compare(quick)
display(summary)

In [ ]:
# Full set (16 questions). Takes a while on CPU.
df, summary = ag.compare()
display(summary)
display(df.groupby(["category", "system"])[["abstain_ok", "source_hit", "has_citation", "keyword_hit", "seconds"]].mean().round(2))
import os, datetime; os.makedirs("data", exist_ok=True)
out = f"data/agentic_vs_plain_{datetime.datetime.now():%Y-%m-%d_%H%M}.csv"; df.to_csv(out, index=False); print("saved", out)

**Where did the two systems behave differently?** Read the `path` column for the agent: it shows *why*.

In [ ]:
wide = df.pivot(index=["id", "category", "question"], columns="system", values=["decision", "abstain_ok", "source_hit"])
display(wide[(wide[("decision", "plain")] != wide[("decision", "agentic")]) | (wide[("source_hit", "plain")] != wide[("source_hit", "agentic")])])

## 8. Audit log
Every agent run is recorded: who (role), what, which steps, which chunks, what decision. No answer text is stored.

In [ ]:
import pandas as pd
log = pd.read_json(ag.AUDIT_PATH, lines=True)
display(log[["ts", "role", "question", "decision", "path", "chunk_ids"]].tail(8))

## 9. How to read the results (and what to tune)
- **`abstain_ok`** should be 1 for unsafe, off-topic, vague and no-access questions, and for answerable ones the system should **answer**.
- **`source_hit`** on cross-source questions is where the agent should beat plain RAG (it searches each source separately).
- **`top_score`**: compare answered and refused questions. If off-topic questions score close to correct ones, `MIN_SCORE` cannot separate them and the grader carries the decision. Set `ag.MIN_SCORE` from your own numbers.
- **`llm_calls` and `seconds`**: the price of control. Report it.
- **`path`**: shows whether the retry was used and whether it helped.

**Next steps for the thesis:** a critic step that removes unsupported sentences, read-only sandbox scenarios (for example a mocked "BGP neighbor down" case), a larger evaluation set, and LangSmith experiments per graph version.